In [1]:
# Interactive map for block 2148496 based on your run_pipeline.py output structure
# - Loads elevation polylines for each trip in sequence
# - Uses deadhead folders for pull_in/pull_out/interline, service folders otherwise
# - Builds a Folium map with ordered polylines and saves it to /mnt/data
#
# If any files are missing on this machine, the script will still create a map and list which
# shapes were missing so you can run it in your environment where the parquet files exist.

import json
from pathlib import Path
from typing import List, Dict

import pandas as pd
import folium

# -----------------
# Inputs (from user)
# -----------------
block_id = 2148496
combined_sequence_json = [
    {"trip_id": "pull_out",   "shape_id": "HTC_52123", "type": "pull_out"},
    {"trip_id": "14687502",   "shape_id": "307560",    "type": "in_service", "start_time": "14:55:00"},
    {"trip_id": "14687522",   "shape_id": "307564",    "type": "in_service", "start_time": "15:54:00"},
    {"trip_id": "14687421",   "shape_id": "307560",    "type": "in_service", "start_time": "17:01:00"},
    {"trip_id": "pull_in",    "shape_id": "58322_HTC", "type": "pull_in"},
]

# -----------------
# Paths (per your run_pipeline.py)
# -----------------
OUT_ROOT = Path("../data/processed")
ELEV_DIR_SERVICE   = OUT_ROOT / "elevation"
ELEV_DIR_DEADHEAD  = OUT_ROOT / "deadhead" / "elevation"

# -----------------
# Helpers
# -----------------
def elev_path_for(shape_id: str, trip_type: str) -> Path:
    """Return the elevation parquet path for the given shape and trip type."""
    if trip_type.lower() in {"pull_in", "pull_out", "interline"}:
        return ELEV_DIR_DEADHEAD / f"elev_{shape_id}.parquet"
    else:
        return ELEV_DIR_SERVICE / f"elev_{shape_id}.parquet"

def read_polyline(path: Path) -> pd.DataFrame:
    """Read a single elevation parquet and return lat/lon sorted by along-route distance if available."""
    df = pd.read_parquet(path)
    # Try to choose sensible order: dist_m if present; otherwise as-is
    order_cols = [c for c in df.columns if c.lower() in ("dist_m", "distance_m", "s_m")]
    if order_cols:
        df = df.sort_values(order_cols[0])
    # Normalize column names we need
    # Expecting columns like: lon, lat, elev_m (based on your pipeline)
    # Fall back if alternative names exist
    lat_col = next((c for c in df.columns if c.lower() in ("lat","latitude","y")), None)
    lon_col = next((c for c in df.columns if c.lower() in ("lon","longitude","x")), None)
    elev_col = next((c for c in df.columns if "elev" in c.lower()), None)
    if lat_col is None or lon_col is None:
        raise ValueError(f"Could not find lat/lon columns in {path.name} (columns: {list(df.columns)})")
    out = df[[lat_col, lon_col] + ([elev_col] if elev_col else [])].copy()
    out.columns = ["lat","lon"] + (["elev_m"] if elev_col else [])
    return out

# Folium color palette to differentiate segments in order
COLORS = ["blue", "red", "green", "purple", "orange", "darkred", "lightred",
          "beige", "darkblue", "darkgreen", "cadetblue", "darkpurple", "pink",
          "lightblue", "lightgreen", "gray", "black", "lightgray"]

# -----------------
# Build map
# -----------------
all_points: List[Dict] = []
missing: List[str] = []

# Try to center on first available shape; otherwise default to Vancouver approx
map_center = [49.2827, -123.1207]
for idx, item in enumerate(combined_sequence_json):
    shape_id = str(item["shape_id"])
    trip_type = item["type"]
    trip_id = item["trip_id"]
    start_time = item.get("start_time")

    path = elev_path_for(shape_id, trip_type)
    if not path.exists():
        missing.append(f"{idx+1}. {trip_type} / trip_id={trip_id} / shape_id={shape_id} → {path}")
        continue

    try:
        df = read_polyline(path)
    except Exception as e:
        missing.append(f"{idx+1}. {trip_type} / trip_id={trip_id} / shape_id={shape_id} → error reading: {e}")
        continue

    # Add to concatenated list with metadata
    for i, row in df.iterrows():
        all_points.append({
            "order": idx+1,
            "trip_type": trip_type,
            "trip_id": trip_id,
            "shape_id": shape_id,
            "lat": float(row["lat"]),
            "lon": float(row["lon"]),
            "elev_m": float(row["elev_m"]) if "elev_m" in df.columns and pd.notna(row["elev_m"]) else None,
            "start_time": start_time
        })

    # set map center from first loaded segment
    if len(all_points) and (idx == 0):
        map_center = [all_points[0]["lat"], all_points[0]["lon"]]

# Create map
m = folium.Map(location=map_center, zoom_start=12, control_scale=True, tiles="OpenStreetMap")

# Draw each segment as its own polyline (ordered)
if all_points:
    pts_df = pd.DataFrame(all_points)
    for ord_idx in sorted(pts_df["order"].unique()):
        seg = pts_df[pts_df["order"] == ord_idx]
        trip_type = seg["trip_type"].iloc[0]
        trip_id = seg["trip_id"].iloc[0]
        shape_id = seg["shape_id"].iloc[0]
        start_time = seg["start_time"].iloc[0]
        color = COLORS[(ord_idx - 1) % len(COLORS)]

        # Build popup/tooltip
        label = f"{ord_idx}. {trip_type.replace('_',' ').title()} — trip_id: {trip_id} — shape_id: {shape_id}"
        if start_time:
            label += f" — start: {start_time}"

        line = [(r["lat"], r["lon"]) for _, r in seg.iterrows()]
        folium.PolyLine(
            line,
            weight=4,
            opacity=0.85,
            color=color,
            tooltip=label
        ).add_to(m)

    # Add start and end markers
    first = pts_df[pts_df["order"] == pts_df["order"].min()].head(1)
    last  = pts_df[pts_df["order"] == pts_df["order"].max()].tail(1)
    if not first.empty:
        folium.Marker(
            [first.iloc[0]["lat"], first.iloc[0]["lon"]],
            tooltip="Start",
            icon=folium.Icon(icon="play", prefix="fa")
        ).add_to(m)
    if not last.empty:
        folium.Marker(
            [last.iloc[0]["lat"], last.iloc[0]["lon"]],
            tooltip="End",
            icon=folium.Icon(icon="flag", prefix="fa")
        ).add_to(m)

# Add a layer control and (if any) a missing-info panel
folium.LayerControl().add_to(m)
if missing:
    miss_html = "<br/>".join(missing)
    folium.map.CustomPane("missing_info").add_to(m)
    folium.Marker(
        location=map_center,
        icon=folium.DivIcon(html=f"<div style='background:white;padding:6px;border:1px solid #ccc;max-width:400px;'>"
                                 f"<b>Missing/Errors ({len(missing)}):</b><br/>{miss_html}</div>")
    ).add_to(m)

# Save map

m.save("block_route_map.html")




In [2]:
import pandas as pd

In [2]:
candidate_map = pd.read_parquet("../data/processed/candidate_stop_map.parquet")

In [3]:
block_summary_depot_only = pd.read_parquet("../data/processed/block_success_summary_depot_only.parquet")

In [4]:
block_summary_depot_only.head()

,block_id,line_group,block_number,service_id,service_day,depot_code,asset_class,asset_class_new,total_distance_km,total_energy_medium_kwh,total_energy_heavy_kwh,avg_kwh_per_km_medium,avg_kwh_per_km_heavy,soc_left_medium_percent,soc_left_heavy_percent,medium_success,heavy_success,combined_sequence_json
0,2159334,22.0,58.0,1,MF,VTC,HY40LF,40-ft,55.755472,86.957634,134.934279,1.559625,2.420108,74.581980,66.075482,SUCCESS,SUCCESS,"[{""trip_id"": ""pull_out"", ""shape_id"": ""VTC_5064..."
1,2159335,22.0,53.0,1,MF,VTC,HY60LF,60-ft,325.495382,533.139210,837.456166,1.637932,2.572867,-4.528229,-58.485136,FAILURE,FAILURE,"[{""trip_id"": ""pull_out"", ""shape_id"": ""VTC_5064..."
2,2159336,22.0,73.0,1,MF,VTC,HY40LF,40-ft,61.109856,99.443811,154.508454,1.627296,2.528372,72.368119,62.604884,SUCCESS,SUCCESS,"[{""trip_id"": ""pull_out"", ""shape_id"": ""VTC_5064..."
3,2159337,22.0,55.0,1,MF,VTC,HY60LF,60-ft,141.553640,237.650316,372.102601,1.678871,2.628704,47.863419,24.024361,SUCCESS,SUCCESS,"[{""trip_id"": ""pull_out"", ""shape_id"": ""VTC_5064..."
4,2159338,22.0,56.0,1,MF,VTC,HY60LF,60-ft,239.645268,395.845908,621.432547,1.651799,2.593135,19.814555,-20.183076,FAILURE,FAILURE,"[{""trip_id"": ""pull_out"", ""shape_id"": ""VTC_5064..."


In [ ]:
block_summary_depot_only = pd.read_parquet("../data/processed/block_success_summary_depot_only.parquet")
df = block_summary_depot_only[(block_summary_depot_only["asset_class_new"] == "40-ft") & (block_summary_depot_only["heavy_success"] == "SUCCESS")].copy()
df['total_distance_km'].sum()

np.float64(58351.50564597333)